# HSK-5 Tutor — train → merge → GGUF (Colab, disk-managed)

End-to-end on a Colab **A100** GPU: QLoRA fine-tune of **Qwen2.5-14B-Instruct** (whatever `config.BASE_MODEL` says), optional before/after eval, merge the adapter, and quantize to a **Q4_K_M GGUF** (~9 GB) you download and serve locally with `app.py`.

**Set the runtime to A100:** *Runtime → Change runtime type → A100 GPU.* The 14B needs it — QLoRA trains in 4-bit (~9 GB), but the **merge** step loads the model in fp16 (~29 GB), which won't fit an L4/T4.

**Why this notebook is structured the way it is — disk.** The intermediates are big: base cache ~29 GB, merged fp16 ~29 GB, f16 GGUF ~29 GB, final Q4 ~9 GB. If they all live at once you blow past Colab's disk (this is what fails). So each heavy step **deletes the previous intermediate as soon as the next step has consumed it**, keeping peak usage ~58 GB. `df -h` prints after each step so you can watch it.

Upload from the repo: `config.py`, `train.py`, `eval.py`, `merge.py`, and `data/train.jsonl` + `data/eval.jsonl`.

## 1. Check the GPU and disk
Confirm you're on an A100 with room to work.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!df -h /content

## 2. Install training + conversion deps
(torch is preinstalled on Colab; this tops up the rest and adds the GGUF-conversion libs.)

In [ ]:
!pip install -q -U "transformers>=4.46" "trl>=0.12,<0.16" "peft>=0.13" \
    "bitsandbytes>=0.44" "accelerate>=1.0" "datasets>=3.0" \
    gguf sentencepiece "protobuf>=5.29.1,<6"

## 3. Upload the project files
Pick all six: `config.py`, `train.py`, `eval.py`, `merge.py`, `train.jsonl`, `eval.jsonl`.
The `.py` are in the repo root; the `.jsonl` are in `data/`. The check below confirms `config.py` is the **14B** one and prints the row counts.

In [ ]:
import os, shutil
from google.colab import files

uploaded = files.upload()
os.makedirs('data', exist_ok=True)
for name in ('train.jsonl', 'eval.jsonl'):
    if name in uploaded:
        shutil.move(name, f'data/{name}')

need = ['config.py', 'train.py', 'eval.py', 'merge.py', 'data/train.jsonl', 'data/eval.jsonl']
missing = [f for f in need if not os.path.exists(f)]
assert not missing, f'missing: {missing}'

base = next(l.strip() for l in open('config.py') if l.strip().startswith('BASE_MODEL'))
print('OK  —', base)
print('train rows:', sum(1 for _ in open('data/train.jsonl')),
      '| eval rows:', sum(1 for _ in open('data/eval.jsonl')))

## 4. Build the GGUF quantizer up front
Clone llama.cpp and build **only** the `llama-quantize` target. `-DGGML_CUDA=OFF` (quantization is CPU-only, so skipping CUDA makes the build much faster) and `-DLLAMA_CURL=OFF` (avoids a libcurl configure error that silently broke earlier runs). Output is shown, and the last line **asserts the binary exists** — do this *before* training so a build problem surfaces now, not after an hour of GPU time.

In [ ]:
!git clone --depth 1 https://github.com/ggml-org/llama.cpp

!cmake -S llama.cpp -B llama.cpp/build -DCMAKE_BUILD_TYPE=Release \
       -DGGML_CUDA=OFF -DLLAMA_CURL=OFF 2>&1 | tail -12
!cmake --build llama.cpp/build --config Release --target llama-quantize -j2 2>&1 | tail -15

import os
QUANT = 'llama.cpp/build/bin/llama-quantize'
assert os.path.exists(QUANT), 'BUILD FAILED — read the output above; fix before continuing'
print('quantizer ready ->', QUANT)

## 5. Sanity run (optional)
~20 steps to confirm it loads and the loss moves. Skip once you trust it.

In [ ]:
!python train.py --max-steps 20

## 6. Full training
~20–30 min on an A100. Saves the LoRA adapter to `outputs/`.

In [ ]:
!python train.py
!echo '--- disk after training ---'; df -h /content

## 7. Before/after eval (optional)
Base vs. tuned on held-out prompts → `outputs/eval_report.md`. Runs **before** the merge because it needs the base cache (which step 8 deletes). Add `--judge` to score with Claude after setting the key.

In [ ]:
# import os; os.environ['ANTHROPIC_API_KEY'] = 'sk-ant-...'   # only for --judge
!python eval.py --n 12

## 8. Merge adapter → fp16, then free the base cache
`merge.py` folds the LoRA into an fp16 copy at `outputs/merged` (~29 GB). Immediately after, delete the ~29 GB HF base cache — it isn't needed again, and this keeps the next step in budget.

In [ ]:
!python merge.py
!rm -rf ~/.cache/huggingface/hub/models--Qwen--Qwen2.5-14B-Instruct
!echo '--- disk after merge (base cache freed) ---'; df -h /content

## 9. Convert merged → f16 GGUF, then free the merged model
Converts `outputs/merged` to a ~29 GB f16 GGUF, then deletes the merged HF folder.

In [ ]:
!python llama.cpp/convert_hf_to_gguf.py outputs/merged \
    --outfile outputs/hsk5-tutor-f16.gguf --outtype f16
!rm -rf outputs/merged
!echo '--- disk after convert (merged freed) ---'; df -h /content

## 10. Quantize → Q4_K_M, then free the f16
The final ~9 GB artifact. Deletes the big f16 GGUF once quantized.

In [ ]:
!./llama.cpp/build/bin/llama-quantize \
    outputs/hsk5-tutor-f16.gguf outputs/hsk5-tutor-q4_k_m.gguf Q4_K_M
!rm outputs/hsk5-tutor-f16.gguf
!echo '--- final artifact ---'; ls -lh outputs/hsk5-tutor-q4_k_m.gguf; df -h /content

## 11. Download the GGUF
Drop it into your local repo's `outputs/` (**back up the current 7B `.gguf` first**), then `python app.py` on the Mac.

In [ ]:
from google.colab import files
files.download('outputs/hsk5-tutor-q4_k_m.gguf')